# Colab GPU Experiment Runner

This notebook is built for shared Google Drive runs and is safe to use in multiple Colab GPU sessions at once.

Default behavior is `MODE = 'auto'`, which means a session will:

- mount Google Drive,
- clone or update the repo into `MyDrive/carbon-aware-recsys`,
- prepare shared caches once,
- claim pending `(category, model)` jobs from the shared `run/` directory, and
- generate final paper plots automatically when the last worker finishes.

In practice, that means you can usually just press `Run all` in each Colab worker session.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = os.environ.get('REPO_URL', 'https://github.com/andersvestrum/carbon-aware-recsys.git')
REPO_BRANCH = os.environ.get('REPO_BRANCH', '6-neumfs-sharp-curve')
REPO_DIRNAME = os.environ.get('REPO_DIRNAME', 'carbon-aware-recsys')
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_ROOT = DRIVE_ROOT / REPO_DIRNAME

# Set this only if you want a different Google Drive checkout path.
MANUAL_PROJECT_ROOT = None

def is_project_root(path):
    return (path / 'scripts' / '05_run_full_experiment.py').exists()

def ensure_repo_checkout(target):
    target = target.expanduser().resolve()
    target.parent.mkdir(parents=True, exist_ok=True)

    if (target / '.git').exists():
        subprocess.run(['git', '-C', str(target), 'fetch', 'origin', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(target), 'checkout', REPO_BRANCH], check=True)
        subprocess.run(['git', '-C', str(target), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
        return target

    if target.exists() and any(target.iterdir()):
        raise FileExistsError(
            f'{target} exists but is not a git checkout. Remove it or set MANUAL_PROJECT_ROOT.'
        )

    if target.exists():
        target.rmdir()

    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(target)],
        check=True,
    )
    return target

if not DRIVE_ROOT.exists():
    raise FileNotFoundError('Google Drive is not mounted. Run the mount cell first.')

PROJECT_ROOT = Path(MANUAL_PROJECT_ROOT).expanduser().resolve() if MANUAL_PROJECT_ROOT else DRIVE_PROJECT_ROOT
PROJECT_ROOT = ensure_repo_checkout(PROJECT_ROOT)
if not is_project_root(PROJECT_ROOT):
    raise FileNotFoundError(f'Repo checkout is missing scripts/05_run_full_experiment.py: {PROJECT_ROOT}')

RUN_DIR = PROJECT_ROOT / 'run'
print({'project_root': str(PROJECT_ROOT), 'run_dir': str(RUN_DIR), 'branch': REPO_BRANCH})
PROJECT_ROOT, RUN_DIR

In [ ]:
%cd {PROJECT_ROOT}
%pip install -q -r requirements.txt

In [ ]:
import torch

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if torch.cuda.is_available()
    else 0.0
)

if GPU_MEMORY_GB >= 35:
    AUTO_TRAIN_BATCH_SIZE = 8192
    AUTO_EVAL_BATCH_SIZE = 16384
    AUTO_USER_BATCH_SIZE = 1024
elif GPU_MEMORY_GB >= 20:
    AUTO_TRAIN_BATCH_SIZE = 6144
    AUTO_EVAL_BATCH_SIZE = 12288
    AUTO_USER_BATCH_SIZE = 768
elif GPU_MEMORY_GB >= 14:
    AUTO_TRAIN_BATCH_SIZE = 4096
    AUTO_EVAL_BATCH_SIZE = 8192
    AUTO_USER_BATCH_SIZE = 512
else:
    AUTO_TRAIN_BATCH_SIZE = 2048
    AUTO_EVAL_BATCH_SIZE = 4096
    AUTO_USER_BATCH_SIZE = 256

print({'gpu': GPU_NAME, 'gpu_memory_gb': round(GPU_MEMORY_GB, 2)})
print({
    'train_batch_size': AUTO_TRAIN_BATCH_SIZE,
    'eval_batch_size': AUTO_EVAL_BATCH_SIZE,
    'user_batch_size': AUTO_USER_BATCH_SIZE,
})

In [ ]:
import os
import socket
import time

MODE = os.environ.get('COLAB_RUN_MODE', 'auto')  # choose: 'auto', 'prepare', 'worker', 'finalize'
WORKER_NAME = os.environ.get('WORKER_NAME') or f"colab-{socket.gethostname()}-{int(time.time())}"

CATEGORIES = ['electronics', 'home_and_kitchen', 'sports_and_outdoors']
MODELS = ['BPR', 'NeuMF', 'LightGCN']

TOP_K_CANDIDATES = 100
TOP_K_RERANK = 10
USER_BATCH_SIZE = AUTO_USER_BATCH_SIZE
TRAIN_BATCH_SIZE = AUTO_TRAIN_BATCH_SIZE
EVAL_BATCH_SIZE = AUTO_EVAL_BATCH_SIZE
LEARNING_RATE = 1e-3
EPOCHS = 50
EVAL_STEP = 10

MAX_USERS = None

PREPARE_CARBON = False
FORCE = False
FORCE_CARBON = False
SKIP_LLM = False
LLM_CACHE_ONLY = False
FINALIZE_WHEN_DONE = MODE == 'auto'

print({
    'mode': MODE,
    'worker_name': WORKER_NAME,
    'categories': CATEGORIES,
    'models': MODELS,
    'run_dir': str(RUN_DIR),
})


In [ ]:
import subprocess
import sys

base_cmd = [
    sys.executable,
    'scripts/05_run_full_experiment.py',
    '--run-dir', str(RUN_DIR),
    '--interim-dir', str(PROJECT_ROOT / 'data' / 'interim'),
    '--top-k-candidates', str(TOP_K_CANDIDATES),
    '--top-k-rerank', str(TOP_K_RERANK),
    '--user-batch-size', str(USER_BATCH_SIZE),
    '--train-batch-size', str(TRAIN_BATCH_SIZE),
    '--eval-batch-size', str(EVAL_BATCH_SIZE),
    '--learning-rate', str(LEARNING_RATE),
    '--epochs', str(EPOCHS),
    '--eval-step', str(EVAL_STEP),
    '--categories', *CATEGORIES,
    '--models', *MODELS,
]

if MAX_USERS is not None:
    base_cmd.extend(['--max-users', str(MAX_USERS)])
if PREPARE_CARBON:
    base_cmd.append('--prepare-carbon')
if FORCE:
    base_cmd.append('--force')
if FORCE_CARBON:
    base_cmd.append('--force-carbon')
if SKIP_LLM:
    base_cmd.append('--skip-llm')
if LLM_CACHE_ONLY:
    base_cmd.append('--llm-cache-only')

def run_step(name, extra_args):
    cmd = base_cmd + extra_args
    print(f'Running {name}:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if MODE == 'auto':
    run_step('prepare', ['--prepare-only'])
    run_step(
        'worker',
        ['--claim-jobs', '--skip-cache-prepare', '--worker-name', WORKER_NAME, '--finalize-when-done'],
    )
elif MODE == 'prepare':
    run_step('prepare', ['--prepare-only'])
elif MODE == 'worker':
    worker_args = ['--claim-jobs', '--skip-cache-prepare', '--worker-name', WORKER_NAME]
    if FINALIZE_WHEN_DONE:
        worker_args.append('--finalize-when-done')
    run_step('worker', worker_args)
elif MODE == 'finalize':
    run_step('finalize', ['--plots-only'])
else:
    raise ValueError(f'Unsupported MODE: {MODE}')

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

summary_path = RUN_DIR / 'results' / 'dataset_summary.csv'
best_points_path = RUN_DIR / 'results' / 'paper_best_pareto_points.csv'
manifest_path = RUN_DIR / 'results' / 'paper_plot_manifest.json'
state_dir = RUN_DIR / 'results' / '_job_state'

if summary_path.exists():
    display(pd.read_csv(summary_path))
if state_dir.exists():
    status_rows = []
    for path in sorted(state_dir.glob('*.done.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'done', **payload})
    for path in sorted(state_dir.glob('*.failed.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'failed', **payload})
    for path in sorted(state_dir.glob('*.running.json')):
        payload = json.loads(path.read_text())
        status_rows.append({'state': 'running', **payload})
    if status_rows:
        display(pd.DataFrame(status_rows).sort_values(['state', 'category', 'model'], na_position='last'))
if best_points_path.exists():
    display(pd.read_csv(best_points_path))
if manifest_path.exists():
    with manifest_path.open() as handle:
        manifest = json.load(handle)
    print(f"Generated {len(manifest['figures'])} figures")
    for figure_path in manifest['figures'][:6]:
        display(Image(filename=figure_path))